In [13]:
#1 setup and import
import duckdb
import pandas as pd
import geopandas as gpd
import leafmap
import ipywidgets as widgets
import os
from IPython.display import display, clear_output

# create an in-memory DuckDB connection and load spatial/httpfs extensions
con = duckdb.connect()
con.install_extension('httpfs')
con.load_extension('httpfs')
con.install_extension('spatial')
con.load_extension('spatial')

In [14]:
#2 load spatial data

TAXI_ZONES_URL = 'https://data.source.coop/cholmes/nyc-taxi-zones/taxi_zones.parquet'
con.sql(f"CREATE OR REPLACE TABLE taxi_zones AS SELECT * FROM '{TAXI_ZONES_URL}'")

if not os.path.exists('nyc_data.db'):
    db_zip_url = 'https://opengeos.org/data/duckdb/nyc_data.db.zip'
    leafmap.download_file(db_zip_url, unzip=True, overwrite=True)

_ = con.execute("ATTACH 'nyc_data.db' AS nyc_data (READ_ONLY)")

In [15]:
#3 process demographics and baselines
con.sql("""
        CREATE OR REPLACE TABLE taxi_zones_utm AS
        SELECT * EXCLUDE (geometry),
            ST_Transform(geometry, 'EPSG:2263', 'EPSG:26918') AS geometry
        FROM taxi_zones
""")

con.sql("""
        CREATE OR REPLACE TABLE zone_demographics AS
        SELECT
            tz.LocationID,
            tz.zone AS TaxiZone,
            tz.borough AS Borough,
            SUM(cb.popn_total) AS TotalPop,
            SUM(cb.popn_white) AS WhitePop,
            SUM(cb.popn_black) AS BlackPop,
            100.0 * SUM(cb.popn_white) / SUM(cb.popn_total) AS white_pct,
            100.0 * SUM(cb.popn_black) / SUM(cb.popn_total) AS black_pct
        FROM nyc_data.nyc_census_blocks AS cb
        JOIN taxi_zones_utm AS tz on ST_Intersects(tz.geometry, cb.geom)
        GROUP BY tz.LocationID, tz.zone, tz.borough
""")

baseline_df = con.sql("""
    SELECT
        ROUND(100.0 * SUM(popn_white) / SUM(popn_total), 2) AS baseline_white_pct,
        ROUND(100.0 * SUM(popn_black) / SUM(popn_total), 2) AS baseline_black_pct
    FROM nyc_data.nyc_census_blocks
""").df()

baseline_white = float(baseline_df['baseline_white_pct'].iloc[0]) / 100.0
baseline_black = float(baseline_df['baseline_black_pct'].iloc[0]) / 100.0

In [ ]:
# 4 process trip data & representative ratios
pu_field = {'FHV': 'PUlocationID', 'Yellow': 'PULocationID'}
do_field = {'FHV': 'DOlocationID', 'Yellow': 'DOLocationID'}
trip_urls = {
    # 'FHV_Jan2025': 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2025-01.parquet',
    'FHV_Feb2025': 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2025-02.parquet',
    # 'FHV_Mar2025': 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2025-03.parquet',
    # 'Yellow_Jan2025': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet',
    'Yellow_Feb2025': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet',
    # 'Yellow_Mar2025': 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-03.parquet',
}

con.execute("CREATE OR REPLACE TABLE trip_counts_pu (service VARCHAR, month VARCHAR, LocationID INTEGER, trips_pu BIGINT)")
con.execute("CREATE OR REPLACE TABLE trip_counts_do (service VARCHAR, month VARCHAR, LocationID INTEGER, trips_do BIGINT)")

for key, url in trip_urls.items():
    service, month = key.split('_')
    pu, do = pu_field[service], do_field[service]
    con.sql(f"""
        INSERT INTO trip_counts_pu
        SELECT
            '{service}',
            '{month}',
            CAST({pu} AS INTEGER),
            COUNT(*)
        FROM '{url}'
        WHERE {pu} IS NOT NULL
            AND CAST({pu} AS INTEGER) NOT IN(0, 264, 265)
        GROUP BY {pu}
    """)
    con.sql(f"""
            INSERT INTO trip_counts_do
            SELECT
                '{service}',
                '{month}',
                CAST({do} AS INTEGER),
                COUNT(*)
            FROM '{url}'
            WHERE {do} IS NOT NULL
                AND CAST({do} AS INTEGER) NOT IN (0, 264, 265)
            GROUP BY {do}
    """)

rr_pu_df = con.sql("""
                    SELECT
                        tp.service,
                        tp.month,
                        SUM(tp.trips_pu * zd.white_pct) * 1.0 / SUM(tp.trips_pu) / {baseline_white * 100} AS RR_white_pu,
                        SUM(tp.trips_pu * zd.black_pct) * 1.0 / SUM(tp.trips_pu) / {baseline_black * 100} AS RR_black_pu
                    FROM trip_counts_pu AS tp
                    JOIN zone_demographics AS zd ON tp.LocationID = zd.LocationID
                    WHERE zd.TotalPop > 0
                    GROUP BY tp.service, tp.month
                """).df()

rr_do_df = con.sql("""
                    SELECT
                        td.service,
                        td.month,
                        SUM(td.trips_do * zd.white_pct) * 1.0 / SUM(td.trips_do) / {baseline_white * 100} AS RR_white_do,
                        SUM(td.trips_do * zd.black_pct) * 1.0 / SUM(td.trips_do) / {baseline_black * 100} AS RR_black_do
                    FROM trip_counts_do AS td
                    JOIN zone_demographics AS zd ON td.LocationID = zd.LocationID
                    WHERE zd.TotalPop > 0
                    GROUP BY td.service, td.month
                """).df()

rr_combined = pd.merge(rr_pu_df, rr_do_df, on=['service', 'month'], how='outer')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HTTPException: HTTP Error: HTTP GET error on 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-03.parquet' (HTTP 403)

In [ ]:
#5 create dashboard!

service_dropdown = widgets.Dropdown(
    options=['FHV', 'Yellow'],
    value='FHV',
    description='Service:',
)

month_dropdown = widgets.Dropdown(
    options=['Jan2025', 'Feb2025', 'Mar2025'],
    value='Feb2025',
    description='Month:',
)

map_output = widgets.Output(
    layout=widgets.Layout(width='50%', height='500px')
)

chart_output = widgets.Output(
    layout=widgets.Layout(width='50%', height='500px')
)

def update_dashboard(change):
    selected_service = service_dropdown.value
    selected_month = month_dropdown.value

    with map_output:
        clear_output(wait=True)
        dynamic_df = con.sql(f"""
            SELECT
                tz.zone AS neighborhood,
                tz.trips_pu AS pickups,
                ST_AsText(ST_Transform(tz.geometry, 'EPSG:26918', 'OGC:CRS84')) AS geometry
            FROM taxi_counts_pu AS tp
            JOIN taxi_zones_utm AS tz ON tp.LocationID = tz.LocationID
            WHERE tp.service = '{selected_service}' AND tp.month = '{selected_month}'""").df()
        
        dynamic_gdf = gpd.GeoDataFrame(
            dynamic_df,
            geometry=gpd.GeoSeries.from_wkt(dynamic_df['geometry']),
            crs='EPSG:4326'
        )

        m = leafmap.Map(center=[40.7, -73.9], zoom=10, draw_control=False)
        m.layout.height = '450px'
        m.add_basemap("CartoDB.Positron")
        m.add_data(
            dynamic_gdf,
            column='pickups',
            cmap='viridis_r',
            layer_name=f'{selected_service} Pickups ({selected_month})'
            layer_name='Pickup Volume'
        )
        display(m)

    with chart_output:
        clear_output(wait=True)
        filtered_rr = rr_combined[
            (rr_combined['service'] == selected_service) &
            (rr_combined['month'] == selected_month)
        ].copy()
        filtered_rr['Month_Service'] = (
            filtered_rr['month'] + ' - ' + filtered_rr['service']
        )

        fig = leafmap.bar_chart(
            data=filtered_rr,
            x="Month_Service",
            y=["RR_white_PU", "RR_black_PU", "RR_white_DO", "RR_black_DO"],
            barmode="group",
            title=f"Relative Risk (RR) for {selected_service} ({selected_month})",
            x_label="",
            y_label="Relative Risk (RR)"
        )

        fig.add_hline(
            y=1.0,
            line_dash="dash",
            line_color="red",
            annotation_text="Equity Baseline (1.0)")
        fig.update_layout(width=500, height=450)
        display(fig)

# link dropdowns to the update function

service_dropdown.observe(update_dashboard, names='value')
month_dropdown.observe(update_dashboard, names='value')

# combine layout and display

dashboard = widgets.VBox([
    widgets.HBox([service_dropdown, month_dropdown]),
    widgets.HBox(
        [map_output, chart_output],
        layout=widgets.Layout(
            width='100%',
            align_items='center'
            )
    )
])

update_dashboard(None)
dashboard